# Week 1 Day 2 assignment: the cheapest model that runs the agent

The challenge from the lab: **with a goal of at least 15 / 25 correct consistently, using the agent, what is the cheapest model you can find?**

This notebook runs the lab's LangChain agent (the same instructions, policy wording and two tools) with a list of cheap OpenRouter models. Each model does the queue of 25 claims twice, because one good run isn't "consistently". A model passes when both runs get at least 15 right.

The models run in parallel, one thread per model. The whole sweep costs well under a dollar, but the slowest models take a minute or more for each claim, so it can run for an hour. Take slow models out of `CANDIDATES`, or set `RUNS = 1`, to finish sooner.

In [ ]:
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
print("OpenRouter API key found" if openrouter_api_key else "Add OPENROUTER_API_KEY to the .env file")

## The data and the prompt, as in the lab

In [ ]:
DATA = Path("../../data")


def load(name):
    return json.loads((DATA / name).read_text(encoding="utf-8"))


claims = load("claims.json")
labels = {decision["claim_id"]: decision for decision in load("evals/decisions.json")}
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")
print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def make_claim_prompt(claim):
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1

## An agent for each model

The lab keeps one global `recorded` dict. Here each run gets its own, so that runs in different threads can't overwrite each other's decisions.

In [ ]:
def make_agent(model, effort):
    """Build the lab's agent for one model, with its own store of recorded decisions."""
    recorded = {}

    @tool
    def get_policy(policy_id: str) -> str:
        """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
        policy = policies.get(policy_id)
        return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"

    @tool
    def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
        """Record the decision on a claim in the claims system.

        payout is the amount to pay in US dollars, or 0 for a decline or a referral.
        rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
        """
        recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
        return f"Recorded {decision} on {claim_id}"

    llm = ChatOpenAI(model=model, base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key,
                     reasoning_effort=effort, timeout=180, max_retries=5)
    agent = create_agent(model=llm, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)
    return agent, recorded


def call_cost(message):
    usage = message.response_metadata.get("token_usage") or {}
    return usage.get("cost") or 0.0


def run_agent(model, effort, run):
    """Run one model's agent over the queue once. A claim with an error or no recorded decision counts as wrong."""
    agent, recorded = make_agent(model, effort)
    results = []
    for claim in claims:
        start = time.time()
        cost, error = 0.0, ""
        try:
            result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]}, {"recursion_limit": 12})
            cost = sum(call_cost(m) for m in result["messages"] if m.type == "ai")
        except Exception as e:
            error = f"{type(e).__name__}: {str(e)[:200]}"
        answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
        label = labels[claim["claim_id"]]
        results.append({"model": model, "effort": effort, "run": run, "claim_id": claim["claim_id"], **answer,
                        "difficulty": label["difficulty"], "right": is_right(answer, label),
                        "cost": cost, "seconds": round(time.time() - start, 1), "error": error})
    right = sum(r["right"] for r in results)
    print(f"{model} run {run}: {right} of {len(results)} right for ${sum(r['cost'] for r in results):.4f}")
    return results

## The candidates

Prices are OpenRouter's, in dollars per million tokens (input / output), as listed in November 2026. Every candidate supports tool calling. Luna at `low` is the baseline from the lab. Each model runs at `low` reasoning; OpenRouter ignores the setting for a model that can't reason.

In [ ]:
CANDIDATES = [
    # model, input $/M, output $/M
    ("openai/gpt-oss-20b", 0.018, 0.09),
    ("qwen/qwen3.7-flash", 0.03, 0.13),
    ("tencent/hy4-flash", 0.035, 0.14),
    ("openai/gpt-5-nano", 0.05, 0.40),
    ("nvidia/nemotron-3-nano-30b-a3b", 0.06, 0.24),
    ("nvidia/nemotron-3.5-lightning", 0.07, 0.20),
    ("mistralai/mistral-small-3.2-24b-instruct", 0.075, 0.20),
    ("google/gemma-4-26b-a4b-it", 0.0765, 0.255),
    ("deepseek/deepseek-v4-flash", 0.09, 0.18),
    ("openai/gpt-oss-120b", 0.09, 0.36),
    ("z-ai/glm-5.3-flash", 0.15, 0.50),
    ("openai/gpt-6-luna", 0.10, 0.50),
]
EFFORT = "low"
RUNS = 2

## Run the sweep

One thread per model; each model's two runs go one after the other, which keeps us under OpenRouter's per-model rate limit for new accounts.

In [ ]:
def run_model(model):
    return [r for run in range(1, RUNS + 1) for r in run_agent(model, EFFORT, run)]


start = time.time()
with ThreadPoolExecutor(max_workers=len(CANDIDATES)) as pool:
    all_results = [r for results in pool.map(run_model, [c[0] for c in CANDIDATES]) for r in results]
print(f"Sweep took {time.time() - start:.0f} seconds")

scored = pd.DataFrame(all_results)
scored.to_csv("results.csv", index=False)

## Results

`worst` is the lower of the two runs, so a model passes when `worst` is at least 15. Cost is the average for a run of the whole queue.

In [ ]:
prices = pd.DataFrame(CANDIDATES, columns=["model", "input $/M", "output $/M"])
per_run = scored.groupby(["model", "run"]).agg(right=("right", "sum"), cost=("cost", "sum"), seconds=("seconds", "mean"),
                                               errors=("error", lambda e: (e != "").sum())).reset_index()
summary = per_run.groupby("model").agg(runs=("right", list), worst=("right", "min"), cost_per_run=("cost", "mean"),
                                        seconds_per_claim=("seconds", "mean"), errors=("errors", "sum")).reset_index()
summary = prices.merge(summary, on="model")
summary["passes"] = summary["worst"] >= 15
summary = summary.sort_values("cost_per_run").reset_index(drop=True)
summary.style.format({"cost_per_run": "${:.4f}", "seconds_per_claim": "{:.1f}"})

In [ ]:
passing = summary[summary["passes"]]
if passing.empty:
    print("No model reached 15 / 25 on both runs")
else:
    best = passing.iloc[0]
    print(f"Cheapest model at 15+ on every run: {best['model']}, scores {best['runs']}, ${best['cost_per_run']:.4f} a run")

Accuracy by difficulty, summed over both runs (out of 2 x the claims at each level).

In [ ]:
scored.pivot_table(index="model", columns="difficulty", values="right", aggfunc="sum").reindex(columns=["routine", "hard", "manual"])

### Where the cheapest passing model went wrong

In [ ]:
def show_misses(results):
    for result in results:
        if not result["right"]:
            label = labels[result["claim_id"]]
            display(Markdown(f"""**{result['claim_id']}, {label['difficulty']}, run {result['run']}**: {result['decision']} ${result['payout']:,.2f}, where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {result['rationale'] or result['error']}
- Handler: {label['reason']}
"""))


if not passing.empty:
    show_misses(scored[scored["model"] == passing.iloc[0]["model"]].to_dict("records"))

## Findings

From one sweep in November 2026: 12 models, two runs of the 25-claim queue each, through the agent at `low` reasoning.

### The models that passed (15 or more on both runs), cheapest first

| Model | Runs | Cost per run | Cost per claim | Seconds per claim |
|---|---|---|---|---|
| openai/gpt-6-luna | 18, 18 | $0.0066 | $0.00026 | 8.8 |
| tencent/hy4-flash | 20, 19 | $0.0087 | $0.00035 | 15.5 |
| z-ai/glm-5.3-flash | 16, 16 | $0.0164 | $0.00066 | 6.4 |
| qwen/qwen3.7-flash | 17, 15 | $0.0193 | $0.00077 | 48.3 |
| deepseek/deepseek-v4-flash | 17, 16 | $0.0249 | $0.00100 | 24.8 |
| google/gemma-4-26b-a4b-it | 16, 16 | $0.0367 | $0.00147 | 27.5 |
| nvidia/nemotron-3-nano-30b-a3b | 15, 16 | $0.0447 | $0.00179 | 30.2 |

Didn't pass: nvidia/nemotron-3.5-lightning (14, 16), openai/gpt-5-nano (11, 11), openai/gpt-oss-20b (10, 10), openai/gpt-oss-120b (9, 11) and mistralai/mistral-small-3.2-24b-instruct (6, 7).

### What I took from it

- **The cheapest model that passed is still Luna at `low`, at $0.0066 a run.** Most of the other models have lower prices per token, but they used more tokens, so they cost more per claim. The price list is not the cost per claim; you have to run the work to find out.
- **Tencent's hy4-flash was the most accurate model: 20 and 19.** It got all 10 hard claims right over the two runs (Luna got 8), for about 1.3 times Luna's cost. Its misses were the manual claims, plus one routine claim. If accuracy matters more than a fraction of a cent, it's the pick.
- **Some failures were with the tools, not the reasoning.** gpt-oss-120b ended 11 claims without recording a decision, and mistral-small ended 19. A model that can't use the tools reliably fails as an agent, however well it reads the policy.
- **Speed varies a lot.** glm-5.3-flash took 6.4 seconds a claim and qwen3.7-flash took 48. qwen3.7-flash also had 5 errors, and nemotron-3.5-lightning had 3.
- **Every model got the manual claims wrong, as the lab expects.** They depend on the claims manual or the customer's history, which the agent doesn't have yet.